# Quick experiment — TopoAE vs CAE: which preserves persistent homology better?

A one-off quick task. It **gates nothing**, writes no verdict artifact, retrains no sealed
fit, and reopens no sealed verdict. It is run *before* phase 02.5-09's open checkpoint is
resolved, as one input to that decision and nothing more.

## The three questions

- **QUICK-TC-01 — does the instrument behave?** A Topological Auto-Encoder is trained to
  minimise exactly the quantity measured here, so it ought to beat a plain auto-encoder at
  its own latent dimension. If it does not, the *measurement* is broken and no conclusion
  about any model follows from the rest of the notebook.
- **QUICK-TC-02 — how bad is the Chart Auto-Encoder, in units that mean something?** Not
  "worse than TopoAE" — by how much, against three references external to both models: a
  plain-AE baseline at the *same* latent dimension, a chance floor from a random latent, and
  an ambient-perturbation ladder that converts an agreement score into an equivalent
  per-point displacement measured in nearest-neighbour spacings.
- **QUICK-TC-03 — does the CAE *invent* structure or *destroy* it?** `loss_x_to_z`
  (destroyed) and `loss_z_to_x` (invented) reported **separately and never summed**, plus
  scale-free directional edge statistics.

## The confound — stated first, because it is disqualifying if ignored

**TopoAE's training objective *is* the metric being scored here.** `topoae.topological_loss`
is what `train_topoae` minimises; `topoae.topological_fidelity` is that same quantity
evaluated on held-out rows. So "TopoAE preserves persistent homology better than a CAE" is
close to **tautological** — it is the model that was trained on the metric, being scored on
the metric. A notebook that prints "TopoAE wins" and stops has answered nothing.

What survives the confound is QUICK-TC-02 and QUICK-TC-03: the **calibrated magnitude** of
the CAE's distortion against references that neither model was trained against, and the
**direction** of that distortion. Those are the results worth reading here. QUICK-TC-01 is
retained only as a known-answer check on the instrument, not as a finding about TopoAE.

## H0 only — what this notebook cannot see

Every persistence quantity here is **0-dimensional**. `topoae.persistence_pairs` returns a
**minimum spanning tree edge set**: connected-component merge structure, and nothing else.
It cannot see loops, voids, or any higher-dimensional feature. No persistence library
(`ripser`, `gudhi`, `persim`, `giotto-tda`) is installed in this environment, and **none is
installed by this work** — §1 probes for all four and prints the result, so the limitation is
evidenced in the output rather than merely asserted in prose. Never read a result below as a
statement about "the topology" without that qualifier. If the question that actually matters
turns out to be H1, this notebook cannot answer it, and that is a finding to write down —
not a licence to install anything.

## What this notebook is forbidden to do

No sealed fit is retrained. Nothing is written into `notebooks/.cache/` — it is opened
read-only, and every cache-writing entry point (`cache.npz_cache`, `cache.json_cache`,
`cache.joblib_cache`, `cae.write_cae_verdict`, `cae.write_cae_handoff`,
`topoae.write_topoae_verdict`, `topoae.write_topoae_handoff`, `topoae.clear_stale_handoff`)
is deliberately never called. No verdict of any kind is produced. The sealed verdicts
`CAE_VERDICT = FAIL` (02.2), the 02.4 TopoAE verdict and `CURVATURE_VERDICT = FAIL` (02.5
stage 1) are prior results referred to in prose only; nothing here revises, softens or
recomputes any of them.

## §1. Environment and provenance

Versions, commit and working directory, then a probe for the four persistence libraries. The
probe exists so the H0-only limitation above is **shown**, not just claimed.

In [1]:
import gc
import json
import subprocess
import sys
import time
from pathlib import Path

# import pu_manifold exactly as the other notebooks do -- relative, never from src/effdim/
NOTEBOOK_DIR = Path.cwd()
assert NOTEBOOK_DIR.name == "notebooks", (
    f"cwd is {NOTEBOOK_DIR!r}, expected the repository's notebooks/ directory. The kernel's "
    "working directory is what pu_manifold's relative import and cache.cache_path both "
    "resolve against; start the kernel in notebooks/ or run nbconvert from there."
)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import torch

from pu_manifold import cache
from pu_manifold import cae
from pu_manifold import topoae

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("=== Reproducibility header (quick: topoae vs cae persistence) ===")
print(f"python         = {sys.version.split()[0]}")
print(f"numpy          = {np.__version__}")
print(f"torch          = {torch.__version__}")
print(f"git commit SHA = {git_sha}")
print(f"cwd            = {NOTEBOOK_DIR}")
print(f"cae module     = {cae.__file__}")
print(f"topoae module  = {topoae.__file__}")

print()
print("=== Persistence-library probe (evidence for the H0-only limitation) ===")
for _mod in ("ripser", "gudhi", "persim", "gtda"):
    try:
        __import__(_mod)
    except ImportError:
        print(f"  {_mod:8s} ABSENT")
    else:
        print(f"  {_mod:8s} present")
print(
    "  -> every persistence quantity in this notebook is 0-dimensional (an MST edge set).\n"
    "     Loops and voids are invisible to it. Nothing is installed to change that here."
)

=== Reproducibility header (quick: topoae vs cae persistence) ===
python         = 3.14.6
numpy          = 2.5.1
torch          = 2.13.0+cpu
git commit SHA = 89de96e
cwd            = /home/akagi/Documents/Projects/EffDim/notebooks
cae module     = /home/akagi/Documents/Projects/EffDim/notebooks/pu_manifold/cae.py
topoae module  = /home/akagi/Documents/Projects/EffDim/notebooks/pu_manifold/topoae.py

=== Persistence-library probe (evidence for the H0-only limitation) ===
  ripser   ABSENT
  gudhi    ABSENT
  persim   ABSENT
  gtda     ABSENT
  -> every persistence quantity in this notebook is 0-dimensional (an MST edge set).
     Loops and voids are invisible to it. Nothing is installed to change that here.


## §2. The two instruments — and why raw fidelity numbers do not compare across latent dimension

**Instrument A — fidelity ratio.** `topoae.topological_fidelity(x, z)` computes the ambient
distance matrix `d_x` unnormalized, rescales the latent by the single global scalar
`topoae.latent_unit_scale(z)` before computing `d_z`, and returns two directional terms:

- `loss_x_to_z` — pairs selected by the **ambient** MST, compared in ambient vs latent
  length. It catches **destroyed** structure.
- `loss_z_to_x` — pairs selected by the **latent** MST, compared in latent vs ambient
  length. It catches **invented** structure.

The function deliberately never returns their sum, because summing collapses two different
failure modes into one number. **They are never summed in this notebook.**
`topoae.t1_gate_value` forms the ratio of each side's `worse = max(loss_x_to_z, loss_z_to_x)`
against a baseline; below 1.0 means better than the baseline.

**Instrument B — scale-free MST edge agreement.** From the ambient MST edge set `E_x` and the
latent MST edge set `E_z` on the same rows:

- `retained = |E_x ∩ E_z| / |E_x|` — the fraction of ambient merges the latent kept.
- `spurious = |E_z \ E_x| / |E_z|` — the fraction of latent merges with no ambient counterpart.
- `jaccard  = |E_x ∩ E_z| / |E_x ∪ E_z|`.

`latent_unit_scale` returns a *single global scalar*, and a global isotropic rescale cannot
reorder any pairwise distance, so it cannot change an MST. Instrument B is therefore exactly
scale-free and carries none of Instrument A's dimension sensitivity.

> **A structural fact about Instrument B that the measurement design did not anticipate, and
> which changes how QUICK-TC-03 must be answered.** Every MST on `n` points has exactly
> `n - 1` edges, so `|E_x| = |E_z|` always, and therefore
> `spurious = (|E_z| - |E_x ∩ E_z|)/|E_z| = 1 - retained` **identically**. `retained` and
> `spurious` are algebraically complementary — they are one number, not two, and the pair
> cannot distinguish "destroyed" from "invented". Both are still reported below (the identity
> is asserted, not assumed), but the invents-versus-destroys question is answered instead by
> Instrument A's two directional terms plus the two **scale-free edge-length asymmetries**
> defined in the next cell, which are genuinely independent of each other:
>
> - `destroyed_stretch = median(d_z on E_x) / median(d_z on E_z)` — how much longer, *in
>   latent units*, a typical ambient-MST edge is than a typical latent-MST edge. Above 1
>   means ambient neighbours were pulled apart: structure **destroyed**.
> - `invented_stretch = median(d_x on E_z) / median(d_x on E_x)` — how much longer, *in
>   ambient units*, a typical latent-MST edge is than a typical ambient-MST edge. Above 1
>   means the latent merged points that were not ambient neighbours: structure **invented**.
>
> Each is a ratio taken *within one space*, so both are scale-free in the same way `retained`
> is, and neither is a function of the other.

**The dimension artifact.** `latent_unit_scale` fixes *mean per-dimension* variance at 1, so
typical latent distances grow like `sqrt(d)` while the ambient scale is fixed — and the
fidelity loss is a **sum of squared absolute length differences**. A raw fidelity value is
therefore largely a report about latent dimension, not about topology. §3 demonstrates this
from measured data rather than asserting it. The rule it forces, which binds the whole
notebook: **every fidelity number is reported only as a ratio against a plain-AE baseline at
the same latent dimension.**

In [2]:
def mst_edge_set(D):
    """0-dimensional persistence pairs of a square symmetric distance array, as a set of
    (i, j) tuples. Thin wrapper over the tested topoae.persistence_pairs."""
    return {(int(i), int(j)) for i, j in topoae.persistence_pairs(np.asarray(D))}


def edge_agreement(x, z, d_x=None, edges_x=None):
    """Instrument B on a row-aligned (ambient, latent) pair. `z` is rescaled by
    topoae.latent_unit_scale first, matching topological_fidelity's own convention (a global
    scalar, so it cannot change an MST -- this is here for convention, not effect)."""
    d_x = topoae.pairwise_distances_f64(x) if d_x is None else d_x
    edges_x = mst_edge_set(d_x.numpy()) if edges_x is None else edges_x
    d_z = topoae.pairwise_distances_f64(z * topoae.latent_unit_scale(z))
    edges_z = mst_edge_set(d_z.numpy())
    kept = edges_x & edges_z
    ix, iz = np.array(sorted(edges_x)).T, np.array(sorted(edges_z)).T
    dz_x, dz_z = d_z[ix[0], ix[1]].numpy(), d_z[iz[0], iz[1]].numpy()
    dx_x, dx_z = d_x[ix[0], ix[1]].numpy(), d_x[iz[0], iz[1]].numpy()
    return {
        "retained": len(kept) / len(edges_x),
        "spurious": len(edges_z - edges_x) / len(edges_z),
        "jaccard": len(kept) / len(edges_x | edges_z),
        "n_edges_x": len(edges_x),
        "n_edges_z": len(edges_z),
        "destroyed_stretch": float(np.median(dz_x) / np.median(dz_z)),
        "invented_stretch": float(np.median(dx_z) / np.median(dx_x)),
    }


def measure(x, z, d_x=None, edges_x=None):
    """Both instruments at once, as one flat dict."""
    return {**topoae.topological_fidelity(x, z), **edge_agreement(x, z, d_x=d_x, edges_x=edges_x)}


RULE = (
    "RULE (binds this whole notebook): a raw topological_fidelity value is not comparable\n"
    "across latent dimensions. Every fidelity number below is reported as a RATIO against a\n"
    "plain-AE baseline at the SAME latent dimension. Instruments B's retained/spurious/\n"
    "jaccard/stretch quantities are scale- and dimension-free and need no such ratio."
)
print("Instrument helpers defined: mst_edge_set, edge_agreement, measure")
print()
print(RULE)

Instrument helpers defined: mst_edge_set, edge_agreement, measure

RULE (binds this whole notebook): a raw topological_fidelity value is not comparable
across latent dimensions. Every fidelity number below is reported as a RATIO against a
plain-AE baseline at the SAME latent dimension. Instruments B's retained/spurious/
jaccard/stretch quantities are scale- and dimension-free and need no such ratio.


## §3. The PU evaluation set — and why it is only a few hundred rows

The CAE (02.2) and the TopoAE (02.4) were trained under **different holdout splits**, seeded
`20260803` and `20260806` respectively. Most of the TopoAE's holdout rows are rows the CAE
*trained on*. Scoring the CAE on the full TopoAE holdout would hand it mostly train-seen
data — a bias in the CAE's favour, on the model this notebook is most sceptical of.

The **primary** evaluation set is therefore the intersection: the rows held out by *both*
models. It is computed here with `np.intersect1d` and its size asserted, never transcribed.
A secondary evaluation on the fuller TopoAE holdout appears in §8 with its leakage count
printed alongside.

This cell closes with §2's dimension-artifact demonstration, measured on these rows.

In [3]:
FIT_KEY = "43cf438bc944c509"
SUBSAMPLE_STEM = "subsample_20260729_a79b3460b838fd0a"
# ONLY amend01-tagged topoae stems are read. The untagged, pre-amendment stems (epochs_run=15)
# also sit on disk: they are the preserved record of a fixed stopping-rule defect and must never
# be reported as the 02.4 result. The tag is written as a literal in every stem below so a
# grep over this notebook's source can prove which family was read.
TOPOAE_SEED = 20260806
CAE_SEEDS = (20260803, 20260804, 20260805)
AMBIENT_DIM = 768
HIDDEN = (250, 250, 250)
ACTIVATION = "silu"
CAE_EMBED_DIM = 40          # the CAE's globally-comparable coordinate (see §8)


def load_npz(stem):
    """Read-only load of a cached archive. cache.cache_path builds the path; nothing in this
    notebook ever writes through the cache helpers."""
    path = cache.cache_path(stem, "npz")
    assert path.exists() and path.stat().st_size > 0, f"missing or empty cached artifact: {path}"
    return dict(np.load(path))


def rebuild_plain_ae(npz, d):
    """The reload/rebuild idiom from notebooks/diagnostics/topoae_evaluate_run.py STEP 1. A
    mismatched constructor would load a shape-mismatched state dict, so the architecture is
    pinned to the cached fits' own."""
    model = cae.PlainAutoEncoder(AMBIENT_DIM, d, hidden=HIDDEN, activation=ACTIVATION)
    model.load_state_dict(cae.arrays_to_state_dict(npz, model.state_dict()))
    model.eval()
    return model


def encode_baseline(d, rows_t, seed=TOPOAE_SEED):
    """Cached plain-AE baselines store weights and y_holdout only -- no z_all -- so they must
    be rebuilt and re-encoded."""
    model = rebuild_plain_ae(load_npz(f"topoae_baseline_{FIT_KEY}_amend01_seed{seed}_d{d}"), d)
    with torch.no_grad():
        return model.encode(rows_t)


X = load_npz(SUBSAMPLE_STEM)["legacysurvey"]
x_all_t = torch.tensor(X, dtype=torch.float32)

_split = load_npz(f"topoae_split_{FIT_KEY}")
topoae_train, topoae_holdout = _split["train_idx"], _split["holdout_idx"]
_cae_fit0 = load_npz(f"cae_fit_{FIT_KEY}_seed{CAE_SEEDS[0]}")
cae_train, cae_holdout = _cae_fit0["train_idx"], _cae_fit0["holdout_idx"]

eval_idx = np.intersect1d(topoae_holdout, cae_holdout)
leaked = np.intersect1d(topoae_holdout, cae_train)

print("=== §3: the PU evaluation set ===")
print(f"ambient array                         : {X.shape}  (L2-normalized onto the unit sphere)")
print(f"TopoAE split (seed 20260806)          : train {topoae_train.size}  holdout {topoae_holdout.size}")
print(f"CAE    split (seed 20260803)          : train {cae_train.size}  holdout {cae_holdout.size}")
print(f"held out by BOTH models (PRIMARY set) : {eval_idx.size}")
print(f"TopoAE-holdout rows the CAE TRAINED on : {leaked.size}"
      f"  ({100 * leaked.size / topoae_holdout.size:.0f}% of the TopoAE holdout)")
print(
    "  -> the two models were trained under different split seeds, so only the intersection\n"
    "     is genuinely held out by both. Scoring the CAE on the full TopoAE holdout would\n"
    "     hand it mostly train-seen rows -- a bias in the CAE's own favour."
)

assert eval_idx.size >= 300, (
    f"primary evaluation set is only {eval_idx.size} rows; below ~300 the MST has too few "
    "edges for the disjoint-half-split null in §7 to say anything at half that size."
)
assert np.intersect1d(eval_idx, cae_train).size == 0, "primary set intersects the CAE training rows"
assert np.intersect1d(eval_idx, topoae_train).size == 0, "primary set intersects the TopoAE training rows"

x_eval = x_all_t[torch.from_numpy(eval_idx)]
d_x_eval = topoae.pairwise_distances_f64(x_eval)
edges_x_eval = mst_edge_set(d_x_eval.numpy())
print(f"\nx_eval {tuple(x_eval.shape)}; ambient MST has {len(edges_x_eval)} edges (= n - 1)")

print()
print("=== §2's rule, demonstrated: raw fidelity scales with latent dimension ===")
print("Same rows, same model family (the cached plain-AE baselines), only d differs.")
print(f"{'baseline':<16}{'loss_x_to_z':>14}{'loss_z_to_x':>14}{'worse':>14}")
print("-" * 58)
for _d in (8, 20, 40):
    _f = topoae.topological_fidelity(x_eval, encode_baseline(_d, x_eval))
    print(f"{'plain AE d=' + str(_d):<16}{_f['loss_x_to_z']:>14.1f}{_f['loss_z_to_x']:>14.1f}{_f['worse']:>14.1f}")
print("-" * 58)
print(
    "latent_unit_scale fixes MEAN PER-DIMENSION variance at 1, so typical latent distances\n"
    "grow like sqrt(d) against a fixed ambient scale, and the loss is a sum of squared\n"
    "absolute length differences. The climb above is dimension, not topology.\n"
)
print(RULE)

=== §3: the PU evaluation set ===
ambient array                         : (10000, 768)  (L2-normalized onto the unit sphere)
TopoAE split (seed 20260806)          : train 8000  holdout 2000
CAE    split (seed 20260803)          : train 8000  holdout 2000
held out by BOTH models (PRIMARY set) : 383
TopoAE-holdout rows the CAE TRAINED on : 1617  (81% of the TopoAE holdout)
  -> the two models were trained under different split seeds, so only the intersection
     is genuinely held out by both. Scoring the CAE on the full TopoAE holdout would
     hand it mostly train-seen rows -- a bias in the CAE's own favour.

x_eval (383, 768); ambient MST has 382 edges (= n - 1)

=== §2's rule, demonstrated: raw fidelity scales with latent dimension ===
Same rows, same model family (the cached plain-AE baselines), only d differs.
baseline           loss_x_to_z   loss_z_to_x         worse
----------------------------------------------------------


plain AE d=8             277.4         225.5         277.4


plain AE d=20            892.9         797.0         892.9


plain AE d=40           1928.9        1746.4        1928.9
----------------------------------------------------------
latent_unit_scale fixes MEAN PER-DIMENSION variance at 1, so typical latent distances
grow like sqrt(d) against a fixed ambient scale, and the loss is a sum of squared
absolute length differences. The climb above is dimension, not topology.

RULE (binds this whole notebook): a raw topological_fidelity value is not comparable
across latent dimensions. Every fidelity number below is reported as a RATIO against a
plain-AE baseline at the SAME latent dimension. Instruments B's retained/spurious/
jaccard/stretch quantities are scale- and dimension-free and need no such ratio.


## §4. Calibration — identity, chance, and an ambient perturbation ladder

A number with no scale is uninterpretable. Three calibrations pin both ends of the range and
supply an external unit:

1. **Identity self-test.** Feeding the ambient array as its own latent must give
   `retained == 1.0` and `jaccard == 1.0` *exactly*. If it does not, the instrument is
   mis-wired and nothing after it means anything.
2. **Chance floor.** A Gaussian random latent of the same dimension on the same rows. This is
   where "no relationship at all" sits.
3. **Ambient perturbation ladder — the external floor.** Displace every ambient point by a
   Euclidean norm of `f × median_nn` and re-measure against the *unperturbed* ambient MST.
   This is what "the same manifold, jittered at resolution scale" costs, and it converts any
   `retained` value into a statement a reader can act on: *this model's H0 agreement is what
   you would get by moving every point about `f` nearest-neighbour spacings.*

   **The scaling trap.** To displace a point by a Euclidean norm of `f × median_nn` in
   `D = 768` dimensions, the per-coordinate Gaussian sigma must be `f × median_nn / sqrt(D)`.
   Perturbing each coordinate by `f × median_nn` directly displaces the point by
   `sqrt(768) × f × median_nn` — about **28× too far**, which silently collapses the ladder
   into noise. The cell therefore prints the **realized** median displacement norm beside its
   nominal multiple and asserts they agree, so the scaling is proved rather than trusted.

In [4]:
CHANCE_DIM = CAE_EMBED_DIM
CHANCE_DRAWS = 5
LADDER_F = (0.25, 0.5, 1.0, 2.0)
LADDER_SEEDS = 5
CALIB_SEED = 20260809

print("=== §4.1 identity self-test ===")
_ident = edge_agreement(x_eval, x_eval, d_x=d_x_eval, edges_x=edges_x_eval)
print(f"retained = {_ident['retained']!r}   spurious = {_ident['spurious']!r}   "
      f"jaccard = {_ident['jaccard']!r}")
assert _ident["retained"] == 1.0 and _ident["jaccard"] == 1.0, (
    "identity self-test failed: passing the ambient array as its own latent did not reproduce "
    "its own MST exactly. The instrument is mis-wired; no number below can be trusted."
)
assert abs(_ident["spurious"] - (1.0 - _ident["retained"])) < 1e-12
print("PASS -- the instrument reproduces an MST exactly when handed the same geometry.")

print()
print("=== §4.2 chance floor: Gaussian random latent, same rows ===")
_rng = np.random.default_rng(CALIB_SEED)
_chance = [
    edge_agreement(x_eval,
                   torch.tensor(_rng.standard_normal((eval_idx.size, CHANCE_DIM)), dtype=torch.float32),
                   d_x=d_x_eval, edges_x=edges_x_eval)
    for _ in range(CHANCE_DRAWS)
]
CHANCE_RETAINED = float(np.median([c["retained"] for c in _chance]))
CHANCE_JACCARD = float(np.median([c["jaccard"] for c in _chance]))
print(f"over {CHANCE_DRAWS} draws at d={CHANCE_DIM}: median retained = {CHANCE_RETAINED:.4f}   "
      f"median jaccard = {CHANCE_JACCARD:.4f}")
print("  -> the bottom of the scale. 1.0 is the top (the identity above).")

print()
print("=== §4.3 ambient perturbation ladder (the external floor) ===")
_dx_np = d_x_eval.numpy().copy()
np.fill_diagonal(_dx_np, np.inf)
MEDIAN_NN = float(np.median(_dx_np.min(axis=1)))
D_AMBIENT = X.shape[1]
print(f"median ambient nearest-neighbour distance = {MEDIAN_NN:.4f}  (ambient D = {D_AMBIENT})")
print(f"per-coordinate sigma = f * median_nn / sqrt(D) = f * {MEDIAN_NN / np.sqrt(D_AMBIENT):.6f}")
print()
print(f"{'f (x median_nn)':>16}{'retained':>12}{'jaccard':>10}{'realized |dx|':>15}{'realized / nn':>15}")
print("-" * 67)
LADDER = []
for _f in LADDER_F:
    _sigma = _f * MEDIAN_NN / np.sqrt(D_AMBIENT)
    _rets, _jacs, _disps = [], [], []
    for _s in range(LADDER_SEEDS):
        _g = np.random.default_rng(CALIB_SEED + 1000 * _s + int(1000 * _f))
        _noise = _g.standard_normal(X[eval_idx].shape) * _sigma
        _disps.append(float(np.median(np.linalg.norm(_noise, axis=1))))
        _pert = torch.tensor(X[eval_idx] + _noise, dtype=torch.float32)
        _a = edge_agreement(x_eval, _pert, d_x=d_x_eval, edges_x=edges_x_eval)
        _rets.append(_a["retained"]); _jacs.append(_a["jaccard"])
    _ret, _jac = float(np.median(_rets)), float(np.median(_jacs))
    _disp = float(np.median(_disps))
    _ratio = _disp / MEDIAN_NN          # realized displacement IN NEAREST-NEIGHBOUR SPACINGS;
                                        # correct scaling makes this equal the nominal f.
    LADDER.append({"f": _f, "retained": _ret, "jaccard": _jac, "disp": _disp, "disp_ratio": _ratio})
    print(f"{_f:>16.2f}{_ret:>12.4f}{_jac:>10.4f}{_disp:>15.4f}{_ratio:>15.3f}")
print("-" * 67)

for _r in LADDER:
    assert abs(_r["disp_ratio"] - _r["f"]) <= 0.20 * _r["f"], (
        f"realized displacement is {_r['disp_ratio']:.3f} nn-spacings, nominal f={_r['f']} -- the "
        "sqrt(D) division in sigma = f * median_nn / sqrt(D) is what breaks if this fails; "
        "without it a point moves sqrt(768) ~ 28x too far and the floor is meaningless."
    )
_ret_seq = [r["retained"] for r in LADDER]
assert all(_ret_seq[i] >= _ret_seq[i + 1] - 1e-12 for i in range(len(_ret_seq) - 1)), (
    f"retained is not non-increasing in f: {_ret_seq}"
)
print("realized displacements match their nominal multiples (within 20%); retained is")
print("non-increasing in f. The ladder is correctly scaled and it discriminates.")
for _r in LADDER:
    print(f"FLOOR f={_r['f']:.2f} retained={_r['retained']:.6f} realized_disp_ratio={_r['disp_ratio']:.6f}")


def equivalent_displacement(retained_value):
    """Report a retained fraction in the ladder's unit: which two rungs, in nearest-neighbour
    spacings, bracket it. The sentence a reader can actually act on."""
    if retained_value > LADDER[0]["retained"]:
        return f"<{LADDER[0]['f']:.2f}x nn"
    for lo, hi in zip(LADDER, LADDER[1:]):
        if retained_value <= lo["retained"] and retained_value >= hi["retained"]:
            return f"{lo['f']:.2f}-{hi['f']:.2f}x nn"
    return f">{LADDER[-1]['f']:.2f}x nn"


print()
print("equivalent_displacement() defined; every model row below carries its reading.")
print(f"  sanity: retained=1.00 -> {equivalent_displacement(1.0)};  "
      f"retained={CHANCE_RETAINED:.3f} (chance) -> {equivalent_displacement(CHANCE_RETAINED)}")

=== §4.1 identity self-test ===
retained = 1.0   spurious = 0.0   jaccard = 1.0
PASS -- the instrument reproduces an MST exactly when handed the same geometry.

=== §4.2 chance floor: Gaussian random latent, same rows ===


over 5 draws at d=40: median retained = 0.0026   median jaccard = 0.0013
  -> the bottom of the scale. 1.0 is the top (the identity above).

=== §4.3 ambient perturbation ladder (the external floor) ===
median ambient nearest-neighbour distance = 0.2511  (ambient D = 768)
per-coordinate sigma = f * median_nn / sqrt(D) = f * 0.009063

 f (x median_nn)    retained   jaccard  realized |dx|  realized / nn
-------------------------------------------------------------------


            0.25      0.9188    0.8499         0.0628          0.250


            0.50      0.8429    0.7285         0.1254          0.499


            1.00      0.6702    0.5039         0.2510          0.999


            2.00      0.3560    0.2166         0.5022          2.000
-------------------------------------------------------------------
realized displacements match their nominal multiples (within 20%); retained is
non-increasing in f. The ladder is correctly scaled and it discriminates.
FLOOR f=0.25 retained=0.918848 realized_disp_ratio=0.249969
FLOOR f=0.50 retained=0.842932 realized_disp_ratio=0.499222
FLOOR f=1.00 retained=0.670157 realized_disp_ratio=0.999312
FLOOR f=2.00 retained=0.356021 realized_disp_ratio=1.999617

equivalent_displacement() defined; every model row below carries its reading.
  sanity: retained=1.00 -> <0.25x nn;  retained=0.003 (chance) -> >2.00x nn


## §5. QUICK-TC-01 — does the instrument behave? The TopoAE ladder against dimension-matched baselines

This is the **known-answer check on the measurement**, not a finding about TopoAE. A
Topological Auto-Encoder is trained to minimise exactly the quantity `topological_fidelity`
reports, so at every latent dimension it ought to beat a plain auto-encoder of matched
capacity trained without the topological term. If it does not, the instrument's behaviour is
in question and **every comparison elsewhere in this notebook inherits that doubt**.

Each rung is scored against its **own** dimension-matched baseline, per §2's rule — the raw
`worse` values climb steeply with `d` and are not comparable across rows.

In [5]:
D_RUNGS = (8, 16, 20, 24, 32, 40)

print("=== §5: TopoAE vs its dimension-matched plain-AE baseline, every rung ===")
print(f"all rungs at seed {TOPOAE_SEED}, on the same {eval_idx.size} primary evaluation rows")
print()
_hdr = (f"{'d':>4}{'topoae worse':>14}{'plain worse':>13}{'ratio':>8}{'topo ret':>10}"
        f"{'plain ret':>11}{'d(ret)':>9}{'topo spur':>11}{'plain spur':>12}"
        f"{'topoae equiv':>16}{'plain equiv':>15}")
print(_hdr)
print("=" * len(_hdr))

ladder_rows = []
for _d in D_RUNGS:
    _zt = torch.tensor(
        load_npz(f"topoae_fit_{FIT_KEY}_amend01_seed{TOPOAE_SEED}_d{_d}")["z_all"][eval_idx],
        dtype=torch.float32,
    )
    _mt = measure(x_eval, _zt, d_x=d_x_eval, edges_x=edges_x_eval)
    _zb = encode_baseline(_d, x_eval)
    _mb = measure(x_eval, _zb, d_x=d_x_eval, edges_x=edges_x_eval)
    _ratio = topoae.t1_gate_value(_mt, _mb)
    ladder_rows.append({"d": _d, "topoae": _mt, "plain": _mb, "ratio": _ratio,
                        "z_topoae": _zt, "z_plain": _zb})
    print(f"{_d:>4}{_mt['worse']:>14.6f}{_mb['worse']:>13.6f}{_ratio:>8.3f}"
          f"{_mt['retained']:>10.3f}{_mb['retained']:>11.3f}"
          f"{_mt['retained'] - _mb['retained']:>+9.3f}"
          f"{_mt['spurious']:>11.3f}{_mb['spurious']:>12.3f}"
          f"{equivalent_displacement(_mt['retained']):>16}"
          f"{equivalent_displacement(_mb['retained']):>15}")
print("=" * len(_hdr))
print("'ratio' = topoae.t1_gate_value(topoae, its own d-matched baseline); < 1.0 = TopoAE better.")
print("The raw 'worse' columns climb with d by the §3 artifact and are NOT comparable down a column.")
print()
for _r in ladder_rows:
    print(f"LADDER d={_r['d']} topoae_worse={_r['topoae']['worse']:.6f} "
          f"plain_worse={_r['plain']['worse']:.6f} ratio={_r['ratio']:.6f} "
          f"topoae_retained={_r['topoae']['retained']:.6f} "
          f"plain_retained={_r['plain']['retained']:.6f}")

# The d=40 rung is the like-for-like partner for the CAE's 40-d embedding (see §8), so its
# objects are bound here for reuse by §6, §7 and §8 rather than being recomputed three times.
_r40 = next(r for r in ladder_rows if r["d"] == 40)
z_topoae40, z_plain40 = _r40["z_topoae"], _r40["z_plain"]
TOPOAE40, BASE40 = _r40["topoae"], _r40["plain"]

_won_fid = [r["d"] for r in ladder_rows if r["ratio"] < 1.0]
_won_ret = [r["d"] for r in ladder_rows if r["topoae"]["retained"] > r["plain"]["retained"]]
print()
print(f"Q1 rungs_won={len(_won_fid)} rungs_total={len(ladder_rows)}")
print()
if len(_won_fid) == len(ladder_rows):
    print(
        f"READ-OUT (QUICK-TC-01): TopoAE beats its dimension-matched baseline on the fidelity\n"
        f"statistic at ALL {len(ladder_rows)} rungs {_won_fid}, and on the independent scale-free\n"
        f"retained fraction at rungs {_won_ret}. The instrument behaves as a known-answer check\n"
        f"requires: the model trained on this objective does better on it than one that was not.\n"
        f"Note the margins are small (ratios {min(r['ratio'] for r in ladder_rows):.3f} to "
        f"{max(r['ratio'] for r in ladder_rows):.3f}) -- the instrument is sensitive enough to\n"
        f"rank them correctly, not so blunt that everything looks identical, and not so sharp\n"
        f"that the topological term looks transformative. This is a check on the ruler, and it\n"
        f"is NOT a result about TopoAE: the confound in the opening cell is exactly that TopoAE\n"
        f"optimised this quantity."
    )
elif _won_fid:
    print(
        f"READ-OUT (QUICK-TC-01), MIXED: TopoAE beats its dimension-matched baseline at only\n"
        f"{len(_won_fid)} of {len(ladder_rows)} rungs ({_won_fid}). The instrument does not\n"
        f"cleanly reproduce the one answer it is supposed to know in advance, so every\n"
        f"comparison elsewhere in this notebook inherits that doubt and should be read as\n"
        f"provisional."
    )
else:
    print(
        "READ-OUT (QUICK-TC-01), FAIL: TopoAE beats its dimension-matched baseline at NO rung.\n"
        "The model trained to minimise this exact quantity does not do better on it than one\n"
        "that ignored it. That is a broken MEASUREMENT, not a result about any model, and no\n"
        "conclusion in the rest of this notebook can be trusted until it is explained."
    )

=== §5: TopoAE vs its dimension-matched plain-AE baseline, every rung ===
all rungs at seed 20260806, on the same 383 primary evaluation rows

   d  topoae worse  plain worse   ratio  topo ret  plain ret   d(ret)  topo spur  plain spur    topoae equiv    plain equiv


   8    246.352820   277.360563   0.888     0.581      0.547   +0.034      0.419       0.453   1.00-2.00x nn  1.00-2.00x nn


  16    652.285983   679.539351   0.960     0.647      0.620   +0.026      0.353       0.380   1.00-2.00x nn  1.00-2.00x nn


  20    881.950931   892.863655   0.988     0.673      0.644   +0.029      0.327       0.356   0.50-1.00x nn  1.00-2.00x nn


  24   1077.315012  1081.456403   0.996     0.657      0.644   +0.013      0.343       0.356   1.00-2.00x nn  1.00-2.00x nn


  32   1483.701712  1552.807567   0.955     0.662      0.626   +0.037      0.338       0.374   1.00-2.00x nn  1.00-2.00x nn


  40   1907.355505  1928.854819   0.989     0.668      0.628   +0.039      0.332       0.372   1.00-2.00x nn  1.00-2.00x nn
'ratio' = topoae.t1_gate_value(topoae, its own d-matched baseline); < 1.0 = TopoAE better.
The raw 'worse' columns climb with d by the §3 artifact and are NOT comparable down a column.

LADDER d=8 topoae_worse=246.352820 plain_worse=277.360563 ratio=0.888204 topoae_retained=0.581152 plain_retained=0.547120
LADDER d=16 topoae_worse=652.285983 plain_worse=679.539351 ratio=0.959894 topoae_retained=0.646597 plain_retained=0.620419
LADDER d=20 topoae_worse=881.950931 plain_worse=892.863655 ratio=0.987778 topoae_retained=0.672775 plain_retained=0.643979
LADDER d=24 topoae_worse=1077.315012 plain_worse=1081.456403 ratio=0.996171 topoae_retained=0.657068 plain_retained=0.643979
LADDER d=32 topoae_worse=1483.701712 plain_worse=1552.807567 ratio=0.955496 topoae_retained=0.662304 plain_retained=0.625654
LADDER d=40 topoae_worse=1907.355505 plain_worse=1928.854819 ratio=0.988

## §6. QUICK-TC-02 and QUICK-TC-03 — the CAE across its three sealed seeds, and the direction of its distortion

First all three sealed 02.2 CAE fits, so no single-seed number stands alone. Then the
directional analysis, which is the most informative result this notebook can produce.

**How the direction is read, stated before the numbers.** Per §2, `retained` and `spurious`
are algebraically complementary and cannot separate the two failure modes. The direction is
therefore taken from two independent, scale-free edge-length asymmetries, each a ratio within
a single space:

- `destroyed_stretch = median(d_z on E_x) / median(d_z on E_z)` — above 1 means ambient
  neighbours were pulled apart in the latent: structure **DESTROYED**.
- `invented_stretch = median(d_x on E_z) / median(d_x on E_x)` — above 1 means the latent
  merged points that were not ambient neighbours: structure **INVENTED**.

Decision rule, fixed here and applied mechanically below: a side counts as active when its
stretch exceeds `STRETCH_TOL = 1.10`; the verdict is `BOTH`, `DESTROYS`, `INVENTS` or
`NEITHER` accordingly, and whichever excess `(stretch − 1)` is larger is named as dominant.
Instrument A's two directional terms are reported beside it as separate baseline-relative
ratios and are **never summed**.

**The standing prior.** Plan 02.5-09 found the CAE fragments the Swiss roll into chart-sized
pieces joined by near-straight chords, which predicts **INVENTS**. Refuting that prior is as
good an outcome as confirming it, and is reported as readily.

In [6]:
STRETCH_TOL = 1.10

print("=== §6.1: the CAE across all three sealed 02.2 seeds (nothing retrained) ===")
_hdr = (f"{'CAE seed':<12}{'loss_x_to_z':>13}{'loss_z_to_x':>13}{'xz/zx':>8}{'ratio vs d40':>14}"
        f"{'retained':>10}{'spurious':>10}{'jaccard':>9}{'equiv. displacement':>22}")
print(_hdr)
print("=" * len(_hdr))
cae_by_seed = {}
cae_z_by_seed = {}
for _s in CAE_SEEDS:
    _z = torch.tensor(load_npz(f"cae_fit_{FIT_KEY}_seed{_s}")["z_all"][eval_idx], dtype=torch.float32)
    _m = measure(x_eval, _z, d_x=d_x_eval, edges_x=edges_x_eval)
    cae_by_seed[_s] = _m
    cae_z_by_seed[_s] = _z
    print(f"{_s:<12}{_m['loss_x_to_z']:>13.1f}{_m['loss_z_to_x']:>13.1f}"
          f"{_m['loss_x_to_z'] / _m['loss_z_to_x']:>8.2f}{topoae.t1_gate_value(_m, BASE40):>14.3f}"
          f"{_m['retained']:>10.3f}{_m['spurious']:>10.3f}{_m['jaccard']:>9.3f}"
          f"{equivalent_displacement(_m['retained']):>22}")
z_cae = cae_z_by_seed[CAE_SEEDS[0]]
print("=" * len(_hdr))
print()
print("across-seed spread over the three sealed CAE fits:")
print(f"  {'statistic':<20}{'min':>12}{'median':>12}{'max':>12}{'max/min':>10}")
print("  " + "-" * 66)
_spread = dict(cae_by_seed)
_spread_stats = [("loss_x_to_z", "{:.1f}"), ("loss_z_to_x", "{:.1f}"), ("retained", "{:.3f}"),
                 ("spurious", "{:.3f}"), ("jaccard", "{:.3f}"),
                 ("destroyed_stretch", "{:.3f}"), ("invented_stretch", "{:.3f}")]
for _k, _fmt in _spread_stats:
    _v = np.array([m[_k] for m in _spread.values()])
    print(f"  {_k:<20}"
          + "".join(_fmt.format(_x).rjust(12) for _x in (_v.min(), np.median(_v), _v.max()))
          + f"{_v.max() / _v.min():>10.2f}")
_v = np.array([topoae.t1_gate_value(m, BASE40) for m in _spread.values()])
print(f"  {'ratio vs d40':<20}"
      + "".join(f"{_x:.3f}".rjust(12) for _x in (_v.min(), np.median(_v), _v.max()))
      + f"{_v.max() / _v.min():>10.2f}")
print("  " + "-" * 66)

_ret = np.array([m["retained"] for m in _spread.values()])
_rank = sorted(CAE_SEEDS, key=lambda s: cae_by_seed[s]["retained"])
_pos = ("the WORST", "the MIDDLE", "the BEST")[_rank.index(CAE_SEEDS[0])]
print()
print(
    f"The CAE's retained fraction is NOT stable across seeds: {_ret.min():.3f} to {_ret.max():.3f}, a factor\n"
    f"of {_ret.max() / _ret.min():.1f}; loss_z_to_x varies by a factor of "
    f"{max(m['loss_z_to_x'] for m in _spread.values()) / min(m['loss_z_to_x'] for m in _spread.values()):.1f}. So no single-seed CAE\n"
    f"number should be quoted as 'the' value -- which is exactly the mistake 02.5-09's chart-\n"
    f"count spread warned about.\n"
    f"\n"
    f"What IS seed-robust is the CONCLUSION, and that is what the spread is here to establish:\n"
    f"every one of the three seeds sits far below the dimension-matched plain-AE baseline\n"
    f"({BASE40['retained']:.3f}), every one reads worse than the 2x-nearest-neighbour rung of the §4 ladder,\n"
    f"and every one sits far above the chance floor ({CHANCE_RETAINED:.3f}) -- so the CAE is badly\n"
    f"degraded, not random. Seed {CAE_SEEDS[0]} carries the directional analysis and the null below\n"
    f"because it is the sealed primary; on retained it is {_pos} of the three, so that choice\n"
    f"is not a favourable one."
)

print()
print("=== §6.2: the directional split -- invents or destroys? ===")


def dir_ratio(name, model_value, baseline_value):
    """Baseline-relative ratio with the same zero/non-finite guard topoae's own gate helpers
    use: raise rather than silently emit inf or nan."""
    if baseline_value == 0.0 or not np.isfinite(baseline_value):
        raise ValueError(f"{name}: baseline quantity is zero or non-finite ({baseline_value!r}) "
                         "-- refusing to divide")
    return float(model_value) / float(baseline_value)


DIRECTIONAL = {
    "CAE embed-40": cae_by_seed[CAE_SEEDS[0]],
    "TopoAE d40": TOPOAE40,
    "plain AE d40": BASE40,
}
_hdr = (f"{'model':<16}{'x_to_z / base':>15}{'z_to_x / base':>15}{'retained':>10}{'spurious':>10}"
        f"{'destroyed_stretch':>19}{'invented_stretch':>18}")
print(_hdr)
print("=" * len(_hdr))
for _name, _m in DIRECTIONAL.items():
    print(f"{_name:<16}"
          f"{dir_ratio('loss_x_to_z', _m['loss_x_to_z'], BASE40['loss_x_to_z']):>15.3f}"
          f"{dir_ratio('loss_z_to_x', _m['loss_z_to_x'], BASE40['loss_z_to_x']):>15.3f}"
          f"{_m['retained']:>10.3f}{_m['spurious']:>10.3f}"
          f"{_m['destroyed_stretch']:>19.3f}{_m['invented_stretch']:>18.3f}")
print("=" * len(_hdr))
print("The two fidelity directions are SEPARATE baseline-relative ratios and are never summed.")
print(f"A stretch above STRETCH_TOL={STRETCH_TOL} marks that failure mode active.")

_cae = DIRECTIONAL["CAE embed-40"]
CAE_XZ_RATIO = dir_ratio("loss_x_to_z", _cae["loss_x_to_z"], BASE40["loss_x_to_z"])
CAE_ZX_RATIO = dir_ratio("loss_z_to_x", _cae["loss_z_to_x"], BASE40["loss_z_to_x"])
_destroyed = _cae["destroyed_stretch"] > STRETCH_TOL
_invented = _cae["invented_stretch"] > STRETCH_TOL
Q3_VERDICT = ("BOTH" if _destroyed and _invented else
              "DESTROYS" if _destroyed else
              "INVENTS" if _invented else "NEITHER")
_dom = ("destroying" if (_cae["destroyed_stretch"] - 1) > (_cae["invented_stretch"] - 1)
        else "inventing")

print()
print(f"READ-OUT (QUICK-TC-03): the CAE {Q3_VERDICT}, dominated by the {_dom} side.")
print(
    f"  destroyed_stretch = {_cae['destroyed_stretch']:.3f}: a typical AMBIENT-MST edge is\n"
    f"    {_cae['destroyed_stretch']:.2f}x longer in the CAE latent than a typical latent-MST edge,\n"
    f"    so points that were ambient neighbours are no longer the ones that merge first.\n"
    f"  invented_stretch  = {_cae['invented_stretch']:.3f}: a typical LATENT-MST edge is\n"
    f"    {_cae['invented_stretch']:.2f}x longer in AMBIENT space than a typical ambient-MST edge,\n"
    f"    so the CAE's own merge structure joins points that were not ambient neighbours.\n"
    f"  For reference the plain-AE d40 baseline sits at "
    f"{BASE40['destroyed_stretch']:.3f} / {BASE40['invented_stretch']:.3f} and the TopoAE d40 at "
    f"{DIRECTIONAL['TopoAE d40']['destroyed_stretch']:.3f} / "
    f"{DIRECTIONAL['TopoAE d40']['invented_stretch']:.3f} -- both essentially 1, i.e. neither\n"
    f"  failure mode active."
)
print()
print(
    f"  Against the 02.5-09 prior (chart fragmentation into pieces joined by near-straight\n"
    f"  chords, predicting INVENTS): the invented side IS active "
    f"({_cae['invented_stretch']:.3f} > {STRETCH_TOL}), so the prior is\n"
    f"  {'CONFIRMED in part' if _invented else 'REFUTED'} -- but it is INCOMPLETE, because the destroyed side is\n"
    f"  active too and is the LARGER of the two. On the PU embedding the CAE is not mainly\n"
    f"  inventing spurious merges; it is mainly failing to keep the real ones. Read that as a\n"
    f"  refinement of 02.5-09 rather than a contradiction of it: fragmentation produces both\n"
    f"  effects at once, and on this data the destruction dominates."
)
print()
print(f"Q3 verdict={Q3_VERDICT} cae_xz_ratio={CAE_XZ_RATIO:.6f} cae_zx_ratio={CAE_ZX_RATIO:.6f} "
      f"cae_retained={_cae['retained']:.6f} cae_spurious={_cae['spurious']:.6f}")

print()
print("--- QUICK-TC-02: how bad, in units that mean something? ---")
print(
    f"  chance floor (no relationship)      retained = {CHANCE_RETAINED:.3f}\n"
    f"  CAE embed-40                        retained = {_cae['retained']:.3f}   "
    f"({equivalent_displacement(_cae['retained'])})\n"
    f"  plain AE d40 (dimension-matched)    retained = {BASE40['retained']:.3f}   "
    f"({equivalent_displacement(BASE40['retained'])})\n"
    f"  TopoAE d40                          retained = "
    f"{DIRECTIONAL['TopoAE d40']['retained']:.3f}   "
    f"({equivalent_displacement(DIRECTIONAL['TopoAE d40']['retained'])})\n"
    f"  identity (perfect)                  retained = 1.000\n"
)
print(
    f"  The CAE is not 'slightly worse than TopoAE'. It is BELOW its own dimension-matched\n"
    f"  plain-AE baseline -- {BASE40['retained'] / _cae['retained']:.1f}x fewer ambient merges kept -- and its H0\n"
    f"  agreement is worse than what you get by displacing every ambient point by more than\n"
    f"  two nearest-neighbour spacings. On Instrument A it nonetheless posts a ratio of\n"
    f"  {topoae.t1_gate_value(_cae, BASE40):.3f}, i.e. it looks like the best model of the three. Those two\n"
    f"  readings cannot both be right, and §6.3 measures which one is misleading."
)

print()
print("=== §6.3: why Instruments A and B disagree about the CAE (measured, not asserted) ===")
_ix = np.array(sorted(edges_x_eval)).T
print(f"{'model':<16}{'mean d_z on E_x':>18}{'mean d_z on E_z':>18}{'mean all-pairs d_z':>20}")
print("-" * 72)
for _name, _z in (("CAE embed-40", z_cae), ("TopoAE d40", z_topoae40), ("plain AE d40", z_plain40)):
    _dz = topoae.pairwise_distances_f64(_z * topoae.latent_unit_scale(_z))
    _ez = np.array(sorted(mst_edge_set(_dz.numpy()))).T
    _tri = np.triu_indices(eval_idx.size, 1)
    print(f"{_name:<16}{_dz[_ix[0], _ix[1]].mean():>18.3f}{_dz[_ez[0], _ez[1]].mean():>18.3f}"
          f"{_dz[_tri].mean():>20.3f}")
print("-" * 72)
print(
    "latent_unit_scale forces all three latents to nearly the same ALL-PAIRS scale, so the\n"
    "rightmost column is near-constant by construction. The middle column is what separates\n"
    "them: the CAE's ambient-MST edges are far SHORTER in its latent than the other two\n"
    "models' are. Instrument A is a sum of squared ABSOLUTE differences against a fixed\n"
    "ambient length, so shorter latent edges mean a smaller penalty -- the CAE scores well on\n"
    "Instrument A for having a compressed local scale, not for preserving which points merge.\n"
    "Instrument B counts the merges themselves and cannot be gamed that way.\n"
    "\n"
    "CONSEQUENCE, and the most transferable finding here: topological_fidelity's baseline-\n"
    "relative ratio -- the 02.4 T1 gate statistic -- can rank a model BEST while its H0 merge\n"
    "structure is near-destroyed. It is safe for comparing models of the same family at the\n"
    "same latent dimension (§5's ladder, where it works), and unsafe across families with\n"
    "different latent scale behaviour. This does not reopen or revise any sealed verdict; it\n"
    "is a limitation of the statistic, recorded here for whoever reads it next."
)

=== §6.1: the CAE across all three sealed 02.2 seeds (nothing retrained) ===
CAE seed      loss_x_to_z  loss_z_to_x   xz/zx  ratio vs d40  retained  spurious  jaccard   equiv. displacement


20260803            780.2        170.8    4.57         0.404     0.183     0.817    0.101             >2.00x nn


20260804            776.8        233.3    3.33         0.403     0.217     0.783    0.122             >2.00x nn


20260805            579.4         48.8   11.86         0.300     0.102     0.898    0.054             >2.00x nn

across-seed spread over the three sealed CAE fits:
  statistic                    min      median         max   max/min
  ------------------------------------------------------------------
  loss_x_to_z                579.4       776.8       780.2      1.35
  loss_z_to_x                 48.8       170.8       233.3      4.78
  retained                   0.102       0.183       0.217      2.13
  spurious                   0.783       0.817       0.898      1.15
  jaccard                    0.054       0.101       0.122      2.27
  destroyed_stretch          1.666       1.938       3.408      2.05
  invented_stretch           1.236       1.261       1.437      1.16
  ratio vs d40               0.300       0.403       0.404      1.35
  ------------------------------------------------------------------

The CAE's retained fraction is NOT stable across seeds: 0.102 to 0.217, a fa

plain AE d40                 3.271             3.140               8.485
------------------------------------------------------------------------
latent_unit_scale forces all three latents to nearly the same ALL-PAIRS scale, so the
rightmost column is near-constant by construction. The middle column is what separates
them: the CAE's ambient-MST edges are far SHORTER in its latent than the other two
models' are. Instrument A is a sum of squared ABSOLUTE differences against a fixed
ambient length, so shorter latent edges mean a smaller penalty -- the CAE scores well on
Instrument A for having a compressed local scale, not for preserving which points merge.
Instrument B counts the merges themselves and cannot be gamed that way.

CONSEQUENCE, and the most transferable finding here: topological_fidelity's baseline-
relative ratio -- the 02.4 T1 gate statistic -- can rank a model BEST while its H0 merge
structure is near-destroyed. It is safe for comparing models of the same family at the
sa

## §7. The resampling null — which gaps are resolved at this sample size?

**Why not "fidelity between two independent subsamples".** `topological_fidelity` and
`topological_loss` are **row-paired**: they index the same rows in both the ambient and the
latent distance matrix. Two disjoint point sets cannot be passed to them at all, and forcing
it would return arithmetic over unrelated rows — a number that looks fine and means nothing.

The construction that does hold is **repeated disjoint half-splits**: partition the
evaluation rows into two disjoint halves, recompute every statistic for every model on each
half, and read the spread off the resulting half-samples. Both halves are cut to exactly the
same size, because the fidelity loss is a sum over `n − 1` edges and is not comparable across
differing point counts.

The tighter and correct comparison is the **paired** per-half difference: TopoAE minus CAE on
the *same* half, which cancels the half-to-half variation common to both. A paired interval
that straddles zero means the gap is **unresolved at this sample size** — reported as such,
not as a result.

In [7]:
R_DRAWS = 20
NULL_SEED = 20260809

_half = eval_idx.size // 2
print("=== §7: disjoint half-split resampling null ===")
print(f"{R_DRAWS} draws x 2 disjoint halves = {2 * R_DRAWS} half-samples, each exactly n={_half} rows")
print(f"(both halves cut to n={_half}; {eval_idx.size - 2 * _half} row dropped per draw when n is odd,")
print(" because the fidelity loss is a sum over n-1 edges and needs a fixed n to compare)")

_null_rng = np.random.default_rng(NULL_SEED)
NULL_MODELS = {"CAE_embed40": z_cae, "TopoAE_d40": z_topoae40, "plain_d40": z_plain40}
null_ret = {k: [] for k in NULL_MODELS}
null_ratio = {k: [] for k in NULL_MODELS}
paired_diff = []

_t0 = time.time()
for _r in range(R_DRAWS):
    _perm = _null_rng.permutation(eval_idx.size)
    for _sel in (_perm[:_half], _perm[_half:2 * _half]):
        _sel_t = torch.from_numpy(_sel)
        _xh = x_eval[_sel_t]
        _dxh = topoae.pairwise_distances_f64(_xh)
        _exh = mst_edge_set(_dxh.numpy())
        _mh = {k: measure(_xh, v[_sel_t], d_x=_dxh, edges_x=_exh) for k, v in NULL_MODELS.items()}
        for _k, _m in _mh.items():
            null_ret[_k].append(_m["retained"])
            null_ratio[_k].append(topoae.t1_gate_value(_m, _mh["plain_d40"]))
        paired_diff.append(_mh["TopoAE_d40"]["retained"] - _mh["CAE_embed40"]["retained"])
print(f"computed in {time.time() - _t0:.1f}s")
print()

_hdr = f"{'model':<14}{'retained p05':>14}{'retained med':>14}{'retained p95':>14}{'ratio med':>12}"
print(_hdr)
print("=" * len(_hdr))
for _k in NULL_MODELS:
    _v = np.array(null_ret[_k])
    print(f"{_k:<14}{np.percentile(_v, 5):>14.4f}{np.median(_v):>14.4f}"
          f"{np.percentile(_v, 95):>14.4f}{np.median(null_ratio[_k]):>12.4f}")
print("=" * len(_hdr))
for _k in NULL_MODELS:
    _v = np.array(null_ret[_k])
    print(f"NULL stat=retained model={_k} p05={np.percentile(_v, 5):.6f} "
          f"med={np.median(_v):.6f} p95={np.percentile(_v, 95):.6f}")

_pd = np.array(paired_diff)
_p05, _med, _p95 = np.percentile(_pd, 5), np.median(_pd), np.percentile(_pd, 95)
_resolved = bool(_p05 > 0 or _p95 < 0)
print()
print(f"paired per-half difference (TopoAE_d40 - CAE_embed40, same half):")
print(f"  p05={_p05:.4f}  median={_med:.4f}  p95={_p95:.4f}")
print(f"NULL_PAIRED diff=topoae_minus_cae p05={_p05:.6f} med={_med:.6f} p95={_p95:.6f} "
      f"resolved={'true' if _resolved else 'false'}")
print()
if _resolved:
    print(
        f"The paired 5-95% interval EXCLUDES zero, so the TopoAE-over-CAE gap in retained\n"
        f"ambient merges is RESOLVED at this sample size: on every one of the {len(_pd)} half-samples\n"
        f"the sign is the same. The gap is a result, not sampling noise."
    )
else:
    print(
        "The paired 5-95% interval STRADDLES zero, so the TopoAE-over-CAE gap is UNRESOLVED at\n"
        "this sample size and must not be reported as a result."
    )
print(
    "\nNote what this null does and does not cover: it is a spread over which ROWS were scored,\n"
    "holding the fitted models fixed. It says nothing about variation across retrainings --\n"
    "for the CAE that is covered separately by the three sealed seeds in §6.1, and for the\n"
    "Swiss roll by the seed sweep in §9."
)

=== §7: disjoint half-split resampling null ===
20 draws x 2 disjoint halves = 40 half-samples, each exactly n=191 rows
(both halves cut to n=191; 1 row dropped per draw when n is odd,
 because the fidelity loss is a sum over n-1 edges and needs a fixed n to compare)


computed in 10.4s

model           retained p05  retained med  retained p95   ratio med
CAE_embed40           0.2053        0.2526        0.3113      0.4262
TopoAE_d40            0.6779        0.7211        0.7545      0.9721
plain_d40             0.6158        0.6684        0.7213      1.0000
NULL stat=retained model=CAE_embed40 p05=0.205263 med=0.252632 p95=0.311316
NULL stat=retained model=TopoAE_d40 p05=0.677895 med=0.721053 p95=0.754474
NULL stat=retained model=plain_d40 p05=0.615789 med=0.668421 p95=0.721316

paired per-half difference (TopoAE_d40 - CAE_embed40, same half):
  p05=0.3945  median=0.4632  p95=0.5168
NULL_PAIRED diff=topoae_minus_cae p05=0.394474 med=0.463158 p95=0.516842 resolved=true

The paired 5-95% interval EXCLUDES zero, so the TopoAE-over-CAE gap in retained
ambient merges is RESOLVED at this sample size: on every one of the 40 half-samples
the sign is the same. The gap is a result, not sampling noise.

Note what this null does and does not cover: it is a spre

## §8. The like-for-like PU comparison

**Which coordinates pair with which.** The CAE's only **globally comparable** coordinate is
its 40-dimensional initial-encoder embedding `z_all`. Its `chart_dim=20` coordinates are
*chart-local*: a pairwise distance between two points assigned to different charts carries no
geometric meaning at all, and `cae.embedding_distortion` raises `ValueError` on exactly that
misuse. So the CAE's comparable representation is **40-d**, and the like-for-like TopoAE rung
is **d=40** — *not* the TopoAE's own primary rung `d=20`. Comparing the 40-d CAE embedding
against a 20-d TopoAE latent would compare latent dimension, not topology (§3). The shared
denominator for both is the cached plain-AE baseline at `d=40`.

The full TopoAE ladder over every rung is §5's job; this section is the matched trio.

In [8]:
print("=== §8: like-for-like PU trio on the PRIMARY evaluation set ===")
print(f"n_eval = {eval_idx.size} rows held out by BOTH models; shared denominator = plain AE d=40")
print()

z_cae = torch.tensor(_cae_fit0["z_all"][eval_idx], dtype=torch.float32)
z_topoae40 = torch.tensor(
    load_npz(f"topoae_fit_{FIT_KEY}_amend01_seed{TOPOAE_SEED}_d40")["z_all"][eval_idx],
    dtype=torch.float32,
)
z_plain40 = encode_baseline(40, x_eval)

TRIO = {
    "CAE embed-40 (seed 20260803)": z_cae,
    "TopoAE d40 (seed 20260806)": z_topoae40,
    "plain AE d40 (baseline)": z_plain40,
}
trio_m = {k: measure(x_eval, v, d_x=d_x_eval, edges_x=edges_x_eval) for k, v in TRIO.items()}
BASE40 = trio_m["plain AE d40 (baseline)"]

_hdr = (f"{'model':<30}{'loss_x_to_z':>12}{'loss_z_to_x':>12}{'worse':>10}{'ratio':>8}"
        f"{'retained':>10}{'spurious':>10}{'jaccard':>9}{'equiv. displacement':>22}")
print(_hdr)
print("=" * len(_hdr))
for _name, _m in trio_m.items():
    _ratio = topoae.t1_gate_value(_m, BASE40)
    print(f"{_name:<30}{_m['loss_x_to_z']:>12.1f}{_m['loss_z_to_x']:>12.1f}{_m['worse']:>10.1f}"
          f"{_ratio:>8.3f}{_m['retained']:>10.3f}{_m['spurious']:>10.3f}{_m['jaccard']:>9.3f}"
          f"{equivalent_displacement(_m['retained']):>22}")
print("=" * len(_hdr))
print(f"{'chance floor (random d40)':<30}{'-':>12}{'-':>12}{'-':>10}{'-':>8}"
      f"{CHANCE_RETAINED:>10.3f}{1 - CHANCE_RETAINED:>10.3f}{CHANCE_JACCARD:>9.3f}"
      f"{equivalent_displacement(CHANCE_RETAINED):>22}")
print("'ratio' is topoae.t1_gate_value against the plain-AE d40 baseline: below 1.0 is better")
print("than the baseline. retained/spurious/jaccard are scale- and dimension-free.")

_cae_m = trio_m["CAE embed-40 (seed 20260803)"]
_topo_m = trio_m["TopoAE d40 (seed 20260806)"]
print()
print(f"TRIO n_eval={eval_idx.size} cae_retained={_cae_m['retained']:.6f} "
      f"topoae_retained={_topo_m['retained']:.6f} plain_retained={BASE40['retained']:.6f} "
      f"cae_ratio={topoae.t1_gate_value(_cae_m, BASE40):.6f} "
      f"topoae_ratio={topoae.t1_gate_value(_topo_m, BASE40):.6f} "
      f"chance_retained={CHANCE_RETAINED:.6f}")

print()
print("--- what the two instruments say about the CAE, and that they DISAGREE ---")
print(
    f"Instrument A ranks the CAE BEST of the three (ratio "
    f"{topoae.t1_gate_value(_cae_m, BASE40):.3f} against the baseline's 1.000).\n"
    f"Instrument B ranks it WORST by a wide margin (retained {_cae_m['retained']:.3f} against\n"
    f"the baseline's {BASE40['retained']:.3f} and the TopoAE's {_topo_m['retained']:.3f}).\n"
    "Both cannot be read as 'preserves H0 better'. The mechanism is measured in §6; the\n"
    "short version is that Instrument A is a sum of squared ABSOLUTE edge-length\n"
    "differences, so a latent that keeps its ambient-neighbour pairs comparatively short --\n"
    "for any reason, including a low-rank collapse -- scores well on it while reordering\n"
    "which points merge first. Instrument B counts the merges themselves and is not\n"
    "foolable that way. Where they disagree, the scale-free instrument is the one to read."
)

=== §8: like-for-like PU trio on the PRIMARY evaluation set ===
n_eval = 383 rows held out by BOTH models; shared denominator = plain AE d=40



model                          loss_x_to_z loss_z_to_x     worse   ratio  retained  spurious  jaccard   equiv. displacement
CAE embed-40 (seed 20260803)         780.2       170.8     780.2   0.404     0.183     0.817    0.101             >2.00x nn
TopoAE d40 (seed 20260806)          1907.4      1766.4    1907.4   0.989     0.668     0.332    0.501         1.00-2.00x nn
plain AE d40 (baseline)             1928.9      1746.4    1928.9   1.000     0.628     0.372    0.458         1.00-2.00x nn
chance floor (random d40)                -           -         -       -     0.003     0.997    0.001             >2.00x nn
'ratio' is topoae.t1_gate_value against the plain-AE d40 baseline: below 1.0 is better
than the baseline. retained/spurious/jaccard are scale- and dimension-free.

TRIO n_eval=383 cae_retained=0.183246 topoae_retained=0.667539 plain_retained=0.628272 cae_ratio=0.404489 topoae_ratio=0.988854 chance_retained=0.002618

--- what the two instruments say about the CAE, and that they 

### §8b. The secondary evaluation set — the full TopoAE holdout, with its bias declared

The 383-row intersection above is the only set genuinely held out by both models, and it is
small. The full 2000-row TopoAE holdout is reported here as a secondary check, with the count
of rows the **CAE trained on** printed first.

The direction of that bias matters: it favours the **CAE**, the model this notebook is most
sceptical of. So a CAE result that is bad on this set is bad *despite* an advantage, which
makes a bad result here stronger evidence than a bad result on the primary set — while a
*good* CAE result here would have to be discounted. The TopoAE and the plain-AE baseline
genuinely held all 2000 rows out, so their numbers carry no such caveat.

In [9]:
print("=== §8b: secondary evaluation on the full TopoAE holdout ===")
print(f"n = {topoae_holdout.size} rows, of which {leaked.size} "
      f"({100 * leaked.size / topoae_holdout.size:.0f}%) are CAE TRAINING rows.")
print("This set is BIASED IN THE CAE'S FAVOUR. A bad CAE result here is bad despite the")
print("advantage; a good one would have to be discounted. TopoAE and the baseline held all")
print("2000 rows out, so their numbers are unaffected.")
print()

_t0 = time.time()
x_sec = x_all_t[torch.from_numpy(topoae_holdout)]
d_x_sec = topoae.pairwise_distances_f64(x_sec)
edges_x_sec = mst_edge_set(d_x_sec.numpy())

_sec_z = {
    "CAE embed-40": torch.tensor(_cae_fit0["z_all"][topoae_holdout], dtype=torch.float32),
    "TopoAE d40": torch.tensor(
        load_npz(f"topoae_fit_{FIT_KEY}_amend01_seed{TOPOAE_SEED}_d40")["z_all"][topoae_holdout],
        dtype=torch.float32),
    "plain AE d40": encode_baseline(40, x_sec),
}
sec_m = {k: measure(x_sec, v, d_x=d_x_sec, edges_x=edges_x_sec) for k, v in _sec_z.items()}
_base_sec = sec_m["plain AE d40"]

_hdr = (f"{'model':<16}{'loss_x_to_z':>13}{'loss_z_to_x':>13}{'ratio':>8}{'retained':>10}"
        f"{'spurious':>10}{'jaccard':>9}{'destroyed_str':>15}{'invented_str':>14}")
print(_hdr)
print("=" * len(_hdr))
for _name, _m in sec_m.items():
    print(f"{_name:<16}{_m['loss_x_to_z']:>13.1f}{_m['loss_z_to_x']:>13.1f}"
          f"{topoae.t1_gate_value(_m, _base_sec):>8.3f}{_m['retained']:>10.3f}"
          f"{_m['spurious']:>10.3f}{_m['jaccard']:>9.3f}"
          f"{_m['destroyed_stretch']:>15.3f}{_m['invented_stretch']:>14.3f}")
print("=" * len(_hdr))
print(f"(computed in {time.time() - _t0:.1f}s at n={topoae_holdout.size}; the n^2 log n MST sort dominates)")
print()
print(f"SECONDARY n_eval={topoae_holdout.size} n_cae_train_leak={leaked.size} "
      f"cae_retained={sec_m['CAE embed-40']['retained']:.6f} "
      f"topoae_retained={sec_m['TopoAE d40']['retained']:.6f} "
      f"plain_retained={_base_sec['retained']:.6f}")
print()
print(
    f"The ordering is unchanged from the primary set: the CAE keeps "
    f"{sec_m['CAE embed-40']['retained']:.3f} of ambient merges\n"
    f"against the baseline's {_base_sec['retained']:.3f} and the TopoAE's "
    f"{sec_m['TopoAE d40']['retained']:.3f}, even with 81% of these rows\n"
    f"seen during CAE training. Note the retained fractions are LOWER for every model here\n"
    f"than at n=383: an MST over five times as many points has five times as many edges to\n"
    f"agree about, and edge agreement is not comparable across different n. Compare down a\n"
    f"column within this table, never across the two tables."
)

=== §8b: secondary evaluation on the full TopoAE holdout ===
n = 2000 rows, of which 1617 (81%) are CAE TRAINING rows.
This set is BIASED IN THE CAE'S FAVOUR. A bad CAE result here is bad despite the
advantage; a good one would have to be discounted. TopoAE and the baseline held all
2000 rows out, so their numbers are unaffected.



model             loss_x_to_z  loss_z_to_x   ratio  retained  spurious  jaccard  destroyed_str  invented_str
CAE embed-40           2260.6        206.3   0.347     0.102     0.898    0.054          2.427         1.384
TopoAE d40             6515.8       5692.2   1.000     0.549     0.451    0.378          1.053         1.037
plain AE d40           6516.5       5529.0   1.000     0.532     0.468    0.362          1.071         1.038
(computed in 30.0s at n=2000; the n^2 log n MST sort dominates)

SECONDARY n_eval=2000 n_cae_train_leak=1617 cae_retained=0.102051 topoae_retained=0.548774 plain_retained=0.531766

The ordering is unchanged from the primary set: the CAE keeps 0.102 of ambient merges
against the baseline's 0.532 and the TopoAE's 0.549, even with 81% of these rows
seen during CAE training. Note the retained fractions are LOWER for every model here
than at n=383: an MST over five times as many points has five times as many edges to
agree about, and edge agreement is not compara